<a href="https://colab.research.google.com/github/dalssam/MBI806B/blob/main/MBI806B_Assignment_270910902.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving clients_cleaned.csv to clients_cleaned.csv
Saving transactions_cleaned.csv to transactions_cleaned.csv


In [2]:
clients = pd.read_csv('clients_cleaned.csv')
tx = pd.read_csv('transactions_cleaned.csv')

print(clients.shape)
print(tx.shape)

(2000, 14)
(50000, 14)


In [3]:
print(tx['transaction_type'].value_counts())

transaction_type
Check    12545
Wire     12513
ACH      12505
SWIFT    12437
Name: count, dtype: int64


In [4]:
# transaction_type risk mapping
# SWIFT: highest, per FATF Recommendation 16 (Travel Rule), which targets
#   international wire transfers specifically due to correspondent banking
#   chain opacity risk
# Wire: second, R.16 also covers domestic wire transfers, but without the
#   cross-border correspondent chain SWIFT specifically carries
# Check: middle, real fraud/kiting typology exists but with a domestic paper trail
# ACH: lowest. Note: ACH as a named system does not exist in New Zealand.
#   This report treats the ACH category as standing in for New Zealand's own
#   domestic batch clearing system, administered by Payments NZ (BECS-based
#   direct credit/direct debit), since no equivalent NZ-specific label exists
#   in the source data.
tx_type_risk_points = {'SWIFT': 100, 'Wire': 70, 'Check': 45, 'ACH': 20}

tx['type_risk_points'] = tx['transaction_type'].map(tx_type_risk_points)

print(tx[['transaction_type', 'type_risk_points']].drop_duplicates())

  transaction_type  type_risk_points
0            Check                45
1            SWIFT               100
3             Wire                70
9              ACH                20


In [5]:
tx['cross_border'] = (tx['client_country'] != tx['counterparty_country']).astype(int)

agg = tx.groupby('client_id').agg(
    tx_count=('transaction_id', 'count'),
    avg_amount=('amount', 'mean'),
    total_amount=('amount', 'sum'),
    mix_score=('type_risk_points', 'mean'),
    pct_structuring=('structuring_pattern_flag', 'mean'),
    pct_rapid=('rapid_movement_flag', 'mean'),
    pct_mispricing=('trade_mispricing_flag', 'mean'),
    pct_ofac_match=('ofac_match_flag', 'mean'),
    pct_cross_border=('cross_border', 'mean'),
    distinct_counterparty_countries=('counterparty_country', 'nunique'),
).reset_index()

print(agg.shape)
print(agg.head())

(2000, 11)
   client_id  tx_count    avg_amount  total_amount  mix_score  \
0          1        26  10045.459231     261181.94  67.500000   
1          2        28   2122.535357      59430.99  57.500000   
2          3        25   2476.962400      61924.06  54.000000   
3          4        29   7249.189655     210226.50  70.172414   
4          5        22   2791.830909      61420.28  62.954545   

   pct_structuring  pct_rapid  pct_mispricing  pct_ofac_match  \
0              0.0   0.038462             0.0        0.000000   
1              0.0   0.000000             0.0        0.000000   
2              0.0   0.040000             0.0        0.880000   
3              0.0   0.068966             0.0        0.034483   
4              0.0   0.000000             0.0        0.000000   

   pct_cross_border  distinct_counterparty_countries  
0          0.846154                               12  
1          0.750000                               10  
2          1.000000                       

In [6]:
df = clients.merge(agg, on='client_id', how='left')
print(df.shape)
print(df['tx_count'].isnull().sum())

(2000, 24)
0


In [7]:
# For each county's fatf_country_flag, ofac_country_flag as dictionary
country_fatf = clients.drop_duplicates('country').set_index('country')['fatf_country_flag'].to_dict()
country_ofac = clients.drop_duplicates('country').set_index('country')['ofac_country_flag'].to_dict()

# counterparty country FATF/OFAC?
tx['counterparty_fatf'] = tx['counterparty_country'].map(country_fatf).fillna(0)
tx['counterparty_ofac'] = tx['counterparty_country'].map(country_ofac).fillna(0)

agg2 = tx.groupby('client_id').agg(
    pct_counterparty_fatf=('counterparty_fatf', 'mean'),
    pct_counterparty_ofac=('counterparty_ofac', 'mean'),
    pct_tx_fatf=('fatf_country_flag', 'mean'),
).reset_index()

df = df.merge(agg2, on='client_id', how='left')
print(df.shape)

(2000, 27)


In [9]:
# Pillar 1: Customer Risk (0-100)
# sanctions_flag and pep_flag are deliberately excluded from this weighted sum.
# Both are handled separately as hard overrides after the composite score is built,
# since they reflect a mandatory compliance response rather than a graduated risk factor.
# See note below for why the two are treated as overrides for different reasons.

client_type_points = {'Individual': 6.25, 'Corporate': 12.5, 'Financial Institution': 18.75, 'NGO': 25}
sector_risk_points = {'Low': 0, 'Medium': 12.5, 'High': 25}

df['p1_client_type'] = df['client_type'].map(client_type_points)
df['p1_sector'] = df['sector_risk'].map(sector_risk_points)
df['p1_sectoral'] = df['sectoral_sanctions_flag'] * 25
df['p1_opacity'] = df['ownership_opacity_score'] * 25

df['customer_risk_score'] = df[['p1_client_type', 'p1_sector',
                                  'p1_sectoral', 'p1_opacity']].sum(axis=1)

print(df['customer_risk_score'].describe())

count    2000.000000
mean       40.915625
std        20.595877
min         6.250000
25%        25.000000
50%        37.500000
75%        56.250000
max       100.000000
Name: customer_risk_score, dtype: float64


In [15]:
# Pillar 2: Country Risk
# Country Risk based on the client's country.

df['p2_client_embargo_override'] = (df['fatf_country_flag'] == 1)
df['country_risk_score'] = ((df['ofac_country_flag'] == 1) & (df['fatf_country_flag'] == 0)).astype(int) * 100

print(df['country_risk_score'].describe())
print("Clients hitting the embargo override:", df['p2_client_embargo_override'].sum())

count    2000.000000
mean       21.100000
std        40.812042
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: country_risk_score, dtype: float64
Clients hitting the embargo override: 179


In [12]:
# Pillar 4: Channel Risk
# Fixed at maximum for every client, per the scenario assumption in the Introduction:
# this report focuses specifically on the bank's digitally onboarded customer segment.
# This is a scenario assumption, not a derived score, so every client gets the same value.
df['channel_risk_score'] = 100.0

In [16]:
df['composite_score'] = (
    0.55 * df['customer_risk_score'] +
    0.30 * df['country_risk_score'] +
    0.15 * df['channel_risk_score']
)

def tier(row):
    if row['sanctions_flag'] == 1:
        return 'High'
    if row['pep_flag'] == 1:
        return 'High'
    if row['p2_client_embargo_override']:
        return 'High'
    if row['composite_score'] >= 71:
        return 'High'
    elif row['composite_score'] >= 41:
        return 'Medium'
    else:
        return 'Low'

df['risk_tier'] = df.apply(tier, axis=1)

review_cycle = {'High': '1 year', 'Medium': '3 years', 'Low': '5 years'}
df['review_cycle'] = df['risk_tier'].map(review_cycle)

print(df['composite_score'].describe())
print(df['risk_tier'].value_counts())

count    2000.000000
mean       43.833594
std        16.868795
min        18.437500
25%        32.187500
50%        42.500000
75%        55.312500
max       100.000000
Name: composite_score, dtype: float64
risk_tier
Low       799
Medium    725
High      476
Name: count, dtype: int64
